# Salutary Duplicity — Exploration

Single simulation on the Zachary karate club graph, 50 epochs, macro (mean-field replicator) strategy update.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# Ensure src is on the path when running without full install
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from salutary_duplicity.model import SalutaryModel, DEFAULT_PARAMS
from salutary_duplicity.networks import (
    make_karate_club,
    assign_random_priors,
    assign_random_strategies,
)

FIG_DIR = '../reports/figures'
os.makedirs(FIG_DIR, exist_ok=True)

print(f'numpy {np.__version__}')
print(f'matplotlib {matplotlib.__version__}')

## Setup

In [ ]:
SEED = 1234
N_EPOCHS = 50

G = make_karate_club()
N = G.number_of_nodes()
print(f'Karate club: N={N}, edges={G.number_of_edges()}')

h = assign_random_priors(G, sigma=0.5, seed=SEED)
strategies = assign_random_strategies(N, p_honest=0.6, seed=SEED)

params = {
    **DEFAULT_PARAMS,
    'update_mode': 'macro',
    'n_sweeps': 50,
    'beta': 1.0,
    'J_strength': 0.5,
    'alpha': 0.05,
}

model = SalutaryModel(G, h, strategies, params=params, seed=SEED)
print(f'Initial fraction honest: {np.mean(strategies):.3f}')

## Run simulation

In [ ]:
history = model.run(N_EPOCHS)

epochs         = [d['epoch'] for d in history]
magnetization  = [d['magnetization'] for d in history]
frac_honest    = [d['fraction_honest'] for d in history]
mean_payoff    = [d['mean_payoff'] for d in history]
payoff_H       = [d['mean_payoff_honest'] for d in history]
payoff_D       = [d['mean_payoff_deceptive'] for d in history]

print(f'Final magnetization:   {magnetization[-1]:.3f}')
print(f'Final fraction honest: {frac_honest[-1]:.3f}')
print(f'Final mean payoff:     {mean_payoff[-1]:.4f}')

## Plot 1: Magnetization over time

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, magnetization, color='steelblue', lw=1.8, label='magnetization m')
ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Magnetization m = mean(s)')
ax.set_title(
    f'Phase A: Magnetization over time\n'
    f'Karate club (N={N}), beta={params["beta"]}, J={params["J_strength"]}, '
    f'n_sweeps={params["n_sweeps"]}, {N_EPOCHS} epochs'
)
ax.legend()
ax.set_ylim(-1.05, 1.05)
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, 'magnetization_over_time.png')
fig.savefig(fig_path, dpi=150)
print(f'Saved: {fig_path}')
plt.show()

## Plot 2: Fraction honest over time

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, frac_honest, color='darkorange', lw=1.8, label='fraction honest (p)')
ax.axhline(0.5, color='k', lw=0.8, ls='--', alpha=0.5, label='p=0.5')
ax.set_xlabel('Epoch')
ax.set_ylabel('Fraction honest')
ax.set_title(
    f'Phase C: Fraction honest over time\n'
    f'Karate club (N={N}), alpha={params["alpha"]}, update_mode={params["update_mode"]!r}, '
    f'{N_EPOCHS} epochs, p0=0.6'
)
ax.legend()
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, 'fraction_honest_over_time.png')
fig.savefig(fig_path, dpi=150)
print(f'Saved: {fig_path}')
plt.show()

## Plot 3: Mean payoff over time (Honest vs Deceptive)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, mean_payoff, color='k', lw=1.8, ls='-', label='overall mean payoff', alpha=0.8)
ax.plot(epochs, payoff_H, color='steelblue', lw=1.5, ls='--', label='Honest agents')
ax.plot(epochs, payoff_D, color='crimson', lw=1.5, ls='--', label='Deceptive agents')
ax.set_xlabel('Epoch')
ax.set_ylabel('Mean payoff')
ax.set_title(
    f'Phase B: Mean payoff over time (H vs D)\n'
    f'Karate club (N={N}), FA={params["FA"]}, C={params["C"]}, gamma={params["gamma"]}, '
    f'{N_EPOCHS} epochs'
)
ax.legend()
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, 'payoff_over_time.png')
fig.savefig(fig_path, dpi=150)
print(f'Saved: {fig_path}')
plt.show()